# EUPE-ViT-B + STM 调制区域分割 — 80 Epochs + Leave-One-Out 交叉验证

基于 `eupe_stm_modulation_segmentation.ipynb`，改为:
- **EPOCHS = 80**（正式训练量）
- **Leave-One-Out (LOO) 交叉验证**：每次留 1 张做验证，其余 N-1 张做训练
- 最终汇报每个 fold 的 best mIoU + 总体平均 mIoU

**预计运行时间**: 20 folds × ~8 min/fold ≈ 2.5-3 小时（GPU），请确保有足够时间。

| class_id | 类别 |
|----------|------|
| 0 | background |
| 1 | modulation_region |
| 2 | sqrt2_modulation_region |


In [ ]:
from __future__ import annotations

import os, random, sys, time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / 'src' / 'lumen').exists():
            return parent
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
if str(REPO_ROOT / 'hyper-data-main' / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'hyper-data-main' / 'src'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SXM_DIR = REPO_ROOT / 'data' / 'stm_dataset' / 'FeTe-sxm'
PNG_DIR = SXM_DIR / 'png'
ANNO_JSON = PNG_DIR / 'project.json'
BACKBONE_PATH = REPO_ROOT / 'checkpoints' / 'EUPE-ViT-B.pt'

IMAGE_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 80  # 正式训练量
LR = 5e-4
WEIGHT_DECAY = 1e-4
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
print(f'Repo root: {REPO_ROOT}')
print(f'EPOCHS: {EPOCHS}')
print(f'SXM dir exists:      {SXM_DIR.exists()}')
print(f'Annotations exist:   {ANNO_JSON.exists()}')
print(f'Backbone exists:     {BACKBONE_PATH.exists()}')

## 1. 加载数据集

In [ ]:
from lumen.data.sxm_stm import (
    SXMSegmentationDataset,
    CLASS_NAMES,
    NUM_CLASSES,
)

# 无增强版本（用于验证和计算 class weights）
dataset = SXMSegmentationDataset(
    sxm_dir=SXM_DIR,
    png_dir=PNG_DIR,
    annotations_json=ANNO_JSON,
    image_size=IMAGE_SIZE,
    augment=False,
)

# 带增强版本（用于训练）
dataset_aug = SXMSegmentationDataset(
    sxm_dir=SXM_DIR,
    png_dir=PNG_DIR,
    annotations_json=ANNO_JSON,
    image_size=IMAGE_SIZE,
    augment=True,
)

N = len(dataset)
print(f'Total samples: {N}')
print(f'Stems: {dataset.stems}')
print(f'\nLOO will run {N} folds, each training on {N-1} samples, validating on 1.')

## 2. 辅助函数

In [ ]:
def collate(batch):
    images = torch.stack([b['image'] for b in batch], dim=0)
    masks = torch.stack([b['mask'] for b in batch], dim=0)
    stems = [b['stem'] for b in batch]
    return {'image': images, 'mask': masks, 'stem': stems}


@torch.no_grad()
def compute_mean_iou(logits: torch.Tensor, mask: torch.Tensor, num_classes: int) -> float:
    pred = logits.argmax(dim=1)
    ious = []
    for c in range(num_classes):
        p = pred == c
        g = mask == c
        inter = (p & g).sum().item()
        union = (p | g).sum().item()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0


@torch.no_grad()
def compute_per_class_iou(logits: torch.Tensor, mask: torch.Tensor, num_classes: int) -> list[float]:
    pred = logits.argmax(dim=1)
    ious = []
    for c in range(num_classes):
        p = pred == c
        g = mask == c
        inter = (p & g).sum().item()
        union = (p | g).sum().item()
        ious.append(inter / union if union > 0 else float('nan'))
    return ious


def compute_class_weights(dataset, train_indices, num_classes, device):
    """计算训练集的逆频率 class weights。"""
    counts = np.zeros(num_classes, dtype=np.int64)
    for i in train_indices:
        flat = dataset[i]['mask'].numpy().ravel()
        for cid in range(num_classes):
            counts[cid] += int((flat == cid).sum())
    freqs = counts / counts.sum()
    inv = 1.0 / np.clip(freqs, 1e-6, None)
    weights = torch.tensor(inv / inv.sum() * num_classes, dtype=torch.float32).to(device)
    return weights

## 3. Leave-One-Out 训练循环

每个 fold：
1. 留出第 `i` 个样本作为验证集
2. 重新初始化模型（同 backbone weights，重新初始化 head）
3. 训练 80 epochs
4. 记录 best mIoU 和最终 mIoU

In [ ]:
from lumen.models.eupe import EUPEEncoder
from lumen.training.downstream import SegmentationTrainer
from lumen.training.losses import SegmentationCriterion

# 存储所有 fold 结果
loo_results = []  # list of dicts: {stem, best_miou, final_miou, per_class_iou, history}

CKPT_DIR = REPO_ROOT / 'artifacts' / 'eupe_stm_seg_loo'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

total_start = time.time()

for fold_idx in range(N):
    val_indices = [fold_idx]
    train_indices = [j for j in range(N) if j != fold_idx]
    val_stem = dataset.stems[fold_idx]
    
    print(f'\n{"=" * 60}')
    print(f'Fold {fold_idx + 1}/{N} — val: {val_stem}')
    print(f'{"=" * 60}')
    
    # --- Class weights (从本 fold 的训练集计算) ---
    class_weights = compute_class_weights(dataset, train_indices, NUM_CLASSES, DEVICE)
    
    # --- 模型初始化 ---
    encoder = EUPEEncoder.from_pretrained(BACKBONE_PATH, strict=False)
    encoder.auto_convert_input_channels = True
    encoder.img_size = IMAGE_SIZE
    
    trainer = SegmentationTrainer(
        encoder=encoder,
        num_classes=NUM_CLASSES,
        trainability='head_only',
        segmentation_head_name='segmentation',
        segmentation_loss='ce',
        scheduler_name='cosine',
        scheduler_t_max=EPOCHS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    ).to(DEVICE)
    
    trainer.criterion = SegmentationCriterion(
        'ce', class_weights=class_weights.detach().cpu(),
    ).to(DEVICE)
    
    # --- DataLoaders ---
    train_dataset = Subset(dataset_aug, train_indices)
    val_dataset = Subset(dataset, val_indices)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, collate_fn=collate)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                            num_workers=0, collate_fn=collate)
    
    # --- 训练 ---
    history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
    best_miou = -1.0
    best_per_class = None
    fold_ckpt = CKPT_DIR / f'best_head_fold{fold_idx:02d}.pt'
    fold_start = time.time()
    
    for epoch in range(EPOCHS):
        trainer.train()
        epoch_losses = []
        for batch in train_loader:
            batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
            log = trainer.train_step(batch)
            epoch_losses.append(log['loss'])
        train_loss = float(np.mean(epoch_losses))
        
        trainer.eval()
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
                logits = trainer(batch['image'])
                val_loss = float(trainer.compute_loss(logits, batch['mask']).item())
                val_miou = compute_mean_iou(logits, batch['mask'], NUM_CLASSES)
                per_class = compute_per_class_iou(logits, batch['mask'], NUM_CLASSES)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_miou'].append(val_miou)
        
        if val_miou > best_miou:
            best_miou = val_miou
            best_per_class = per_class
            torch.save({'head_state_dict': trainer.head.state_dict(),
                        'epoch': epoch, 'val_miou': val_miou, 'fold': fold_idx},
                       fold_ckpt)
    
    fold_time = time.time() - fold_start
    final_miou = history['val_miou'][-1]
    
    print(f'  Done in {fold_time:.0f}s | best mIoU={best_miou:.4f} (final={final_miou:.4f})')
    print(f'  Per-class IoU (best): {["{:.3f}".format(x) for x in best_per_class]}')
    
    loo_results.append({
        'fold': fold_idx,
        'stem': val_stem,
        'best_miou': best_miou,
        'final_miou': final_miou,
        'per_class_iou': best_per_class,
        'history': history,
    })
    
    # 释放显存
    del trainer, encoder
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f'\n\nAll {N} folds complete in {total_time / 60:.1f} minutes.')

## 4. 结果汇总

In [ ]:
print(f'{"Fold":>4s}  {"Stem":>12s}  {"Best mIoU":>10s}  {"bg IoU":>8s}  {"mod IoU":>8s}  {"sqrt2 IoU":>10s}')
print('-' * 70)

best_mious = []
per_class_all = []  # (N, 3)

for r in loo_results:
    pci = r['per_class_iou']
    per_class_all.append(pci)
    best_mious.append(r['best_miou'])
    pci_str = [f'{x:.3f}' if not np.isnan(x) else '  N/A' for x in pci]
    print(f"{r['fold']+1:4d}  {r['stem']:>12s}  {r['best_miou']:10.4f}  "
          f"{pci_str[0]:>8s}  {pci_str[1]:>8s}  {pci_str[2]:>10s}")

mean_miou = float(np.mean(best_mious))
std_miou = float(np.std(best_mious))
per_class_mean = np.nanmean(per_class_all, axis=0)

print('-' * 70)
print(f'Mean mIoU: {mean_miou:.4f} ± {std_miou:.4f}')
print(f'Per-class mean IoU: bg={per_class_mean[0]:.3f}, mod={per_class_mean[1]:.3f}, sqrt2={per_class_mean[2]:.3f}')
print(f'\nMin mIoU fold: {loo_results[np.argmin(best_mious)]["stem"]} ({min(best_mious):.4f})')
print(f'Max mIoU fold: {loo_results[np.argmax(best_mious)]["stem"]} ({max(best_mious):.4f})')

## 5. 训练曲线（所有 fold 叠加）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for r in loo_results:
    epochs_arr = range(1, EPOCHS + 1)
    axes[0].plot(epochs_arr, r['history']['train_loss'], alpha=0.3, color='blue')
    axes[1].plot(epochs_arr, r['history']['val_loss'], alpha=0.3, color='orange')
    axes[2].plot(epochs_arr, r['history']['val_miou'], alpha=0.3, color='green')

# 画平均曲线
mean_train = np.mean([r['history']['train_loss'] for r in loo_results], axis=0)
mean_val_loss = np.mean([r['history']['val_loss'] for r in loo_results], axis=0)
mean_val_miou = np.mean([r['history']['val_miou'] for r in loo_results], axis=0)

epochs_arr = range(1, EPOCHS + 1)
axes[0].plot(epochs_arr, mean_train, color='blue', linewidth=2, label='mean')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].set_title('Train Loss (all folds)')
axes[0].legend()

axes[1].plot(epochs_arr, mean_val_loss, color='orange', linewidth=2, label='mean')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss'); axes[1].set_title('Val Loss (all folds)')
axes[1].legend()

axes[2].plot(epochs_arr, mean_val_miou, color='green', linewidth=2, label='mean')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val mIoU'); axes[2].set_title('Val mIoU (all folds)')
axes[2].legend()

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'loo_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved training curves to: {CKPT_DIR / "loo_training_curves.png"}')

## 6. mIoU 分布

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
stems = [r['stem'] for r in loo_results]
mious = [r['best_miou'] for r in loo_results]

bars = ax.bar(range(N), mious, color='steelblue', edgecolor='black', alpha=0.8)
ax.axhline(mean_miou, color='red', linestyle='--', label=f'mean={mean_miou:.3f}')
ax.set_xticks(range(N))
ax.set_xticklabels(stems, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Validation Sample')
ax.set_ylabel('Best mIoU')
ax.set_title(f'Leave-One-Out mIoU ({EPOCHS} epochs, {N} folds)')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'loo_miou_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. 结论与建议

- **如果 mean mIoU > 0.5**：模型对调制区域有较好区分能力，可投入预标注使用
- **如果 std 很大**（>0.15）：说明某些特定图像的标注或数据质量可能有问题，检查 min fold 的原图
- **如果 mIoU 饱和早**（看平均曲线 20 epoch 后不涨）：80 epochs 够了；否则可试 120
- **如果过拟合严重**（train loss 持续降但 val loss 回升）：考虑更小的 head 或更强的增强

LOO 结果对比之前 80/20 split（单次 20 epochs, best mIoU~0.44）应该给出更稳定的评估。